In [1]:
import random
import sys
import json
import csv
import os
import threading
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
from ltl_stats_client import LTLStats, LTLStatsError
from time import time

alphabet = ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z']
unary = ['!', 'X ']
binary_non_temporal = ['&&', '||']
binary_temporal = ['U','W']


In [2]:
P = 1/4
weights = [(1-P)/4]*4 + [P/2]*2

In [3]:
# Sweep parameters
N_VALUES = [1, 2, 3]
L_VALUES = [5, 10, 15, 20]
F = 100  # number of random formulas per (N, L) combination

MTSA_DIR = "/home/Lakso/Programas/mtsa/maven-root/mtsa"

# Per-formula wall-clock limit. A formula that takes longer is abandoned (its
# JVM is killed), recorded as timed out, and the batch moves on.
TIMEOUT_S = 15 * 60

# Output files
FORMULAS_FILE = "formulas.txt"
STATS_FILE = "stats.csv"
RESULTS_FILE = "results.json"
TIMEOUTS_FILE = "timeouts.txt"

# generate_formula() reads N from the global scope; it is reset inside the sweep
N = N_VALUES[0]

In [4]:
def generate_formula(L):
    if L == 1:
        return random.choice(alphabet[:N])

    op = random.choices(unary+binary_temporal+binary_temporal, weights)[0]
    if L == 2:
        op = random.choice(unary)

    if op in unary:
        return '('+op+generate_formula(L-1)+')'

    S = random.randrange(1,L-1)

    return '(' + generate_formula(S) + ' ' + op + ' ' + generate_formula(L-S-1) + ')'


In [5]:
# Run every (N, L) combination, collecting automaton stats and timings.
#
# LTLStatsService is single-threaded: it holds a class monitor for the whole
# compilation, so one JVM never uses more than one core. To use the whole CPU we
# start several independent JVMs (one per worker) and spread each combination's
# formulas across them. The pool is created once and reused for the whole sweep.
#
# Memory: without an -Xmx cap each JVM's max heap is ergonomic (~25% of RAM, so
# ~4 GB on a 15 GB box). A few workers hitting large-L formulas at once then blow
# past physical RAM and the Linux OOM killer takes down the kernel (and half the
# desktop session). Cap each JVM's heap and keep N_WORKERS * heap well under RAM;
# ExitOnOutOfMemoryError makes a JVM that still overflows die cleanly (recorded
# as "worker crashed") instead of thrashing swap.
os.environ["JAVA_TOOL_OPTIONS"] = "-Xmx2g -XX:+ExitOnOutOfMemoryError"

# 3 workers * 2 GB heap + JVM overhead ~= 8 GB peak, safe on a 15 GB machine.
# Raise -Xmx and drop this to 2 if specific formulas legitimately need more heap.
N_WORKERS = min(3, max(1, (os.cpu_count() or 4) - 2))

all_results = []      # one record per formula
combo_times = {}      # (N, L) -> wall-clock seconds for that whole combination

_worker = threading.local()   # one LTLStats subprocess per pool thread
_svcs = []                     # every spawned subprocess, for teardown
_svcs_lock = threading.Lock()


def _get_svc():
    svc = getattr(_worker, "svc", None)
    if svc is None:
        svc = LTLStats.spawn(MTSA_DIR)
        with _svcs_lock:
            _svcs.append(svc)
        _worker.svc = svc
    return svc


def _blank_rec(n, L, i, formula, *, ok, error, timed_out):
    return {
        "N": n, "L": L, "index": i, "formula": formula,
        "ok": ok, "error": error, "is_property": None,
        "gba_states": None, "gba_transitions": None,
        "buchi_states": None, "buchi_transitions": None,
        "timed_out": timed_out,
    }


def _run_formula(task):
    n, L, i, formula = task
    svc = _get_svc()
    run_start = time()
    # svc.stats() blocks on a pipe read, so it can't be cancelled in place.
    # Run it in a helper thread and give up on it after TIMEOUT_S; killing the
    # JVM unblocks that thread (its read returns EOF and raises).
    call_ex = ThreadPoolExecutor(max_workers=1)
    try:
        fut = call_ex.submit(svc.stats, formula, raise_on_error=False)
        try:
            res = fut.result(timeout=TIMEOUT_S)
        except FuturesTimeout:
            svc.kill()             # abort the hung compilation
            _worker.svc = None     # a fresh JVM is spawned on the next task
            print(f"  TIMEOUT N={n} L={L} #{i} after {TIMEOUT_S}s: {formula}")
            rec = _blank_rec(n, L, i, formula, ok=False,
                             error=f"timed out after {TIMEOUT_S}s", timed_out=True)
        else:
            rec = {
                "N": n, "L": L, "index": i, "formula": formula,
                "ok": res.ok, "error": res.error, "is_property": res.is_property,
                "gba_states": res.gba.states if res.gba else None,
                "gba_transitions": res.gba.transitions if res.gba else None,
                "buchi_states": res.buchi.states if res.buchi else None,
                "buchi_transitions": res.buchi.transitions if res.buchi else None,
                "timed_out": False,
            }
    except LTLStatsError as e:
        _worker.svc = None  # this JVM died; a fresh one is spawned on the next task
        rec = _blank_rec(n, L, i, formula, ok=False,
                         error=f"worker crashed: {e}", timed_out=False)
    finally:
        call_ex.shutdown(wait=False)
    rec["run_time_s"] = time() - run_start
    return rec


sweep_start = time()
try:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        for n in N_VALUES:
            N = n  # used by generate_formula
            for L in L_VALUES:
                formulas = [generate_formula(L) for _ in range(F)]
                tasks = [(n, L, i, formula) for i, formula in enumerate(formulas)]
                combo_start = time()
                combo_results = list(ex.map(_run_formula, tasks))  # results stay in order
                combo_elapsed = time() - combo_start
                all_results.extend(combo_results)
                combo_times[(n, L)] = combo_elapsed
                n_to = sum(r["timed_out"] for r in combo_results)
                print(f"N={n} L={L}: {F} formulas in {combo_elapsed:.4f}s"
                      + (f" ({n_to} timed out)" if n_to else ""))
        total_elapsed = time() - sweep_start
finally:
    # Always tear the JVMs down -- otherwise an interrupt or exception orphans
    # them and they keep holding gigabytes until the machine is rebooted.
    for _svc in _svcs:
        try:
            _svc.close()
        except Exception:
            pass
        try:
            _svc.kill()
        except Exception:
            pass

n_timeouts = sum(r["timed_out"] for r in all_results)
print(f"Terminado en {total_elapsed}s ({n_timeouts} formulas timed out)")

building project + classpath (4b119bf70f3fd56bdea91072c5becc8eb9967873-dirty)...
building project + classpath (4b119bf70f3fd56bdea91072c5becc8eb9967873-dirty)...
building project + classpath (4b119bf70f3fd56bdea91072c5becc8eb9967873-dirty)...
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError
building project + classpath (4b119bf70f3fd56bdea91072c5becc8eb9967873-dirty)...
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError
Error: Could not find or load main class ltsa.lts.ltl.LTLStatsService
Caused by: java.lang.ClassNotFoundException: ltsa.lts.ltl.LTLStatsService
building project + classpath (4b119bf70f3fd56bdea91072c5becc8eb9967873-dirty)...
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError
building project + classpath (4b119bf70f3fd56bdea91072c5becc8eb9967

N=1 L=5: 100 formulas in 10.8821s
N=1 L=10: 100 formulas in 0.8234s


building project + classpath (4b119bf70f3fd56bdea91072c5becc8eb9967873-dirty)...
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError


  TIMEOUT N=1 L=15 #83 after 900s: ((!(!a)) W (!((a W a) W ((X a) W (a U a)))))
N=1 L=15: 100 formulas in 1336.8479s (1 timed out)


building project + classpath (4b119bf70f3fd56bdea91072c5becc8eb9967873-dirty)...
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError


...

  TIMEOUT N=3 L=20 #91 after 900s: ((X a) U (X ((((!(c W (!a))) W (b W (X a))) W (a W b)) W a)))


building project + classpath (4b119bf70f3fd56bdea91072c5becc8eb9967873-dirty)...
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError


  TIMEOUT N=3 L=20 #94 after 900s: ((!((b U c) W (a W b))) U ((X ((!c) W ((b U a) W b))) U a))


building project + classpath (4b119bf70f3fd56bdea91072c5becc8eb9967873-dirty)...
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError
Picked up JAVA_TOOL_OPTIONS: -Xmx2g -XX:+ExitOnOutOfMemoryError


  TIMEOUT N=3 L=20 #95 after 900s: ((X b) W (((X c) U (X (!(X ((!c) W c))))) U ((X b) W (a U c))))
  TIMEOUT N=3 L=20 #97 after 900s: (((X b) W c) U (((X c) U (!c)) U (!((!a) U ((a U c) W c)))))
N=3 L=20: 100 formulas in 15534.9278s (47 timed out)
Terminado en 36314.95319890976s (94 formulas timed out)


In [6]:
# Save the generated formulas (N, L, index, formula) to a file.
with open(FORMULAS_FILE, "w") as fh:
    for rec in all_results:
        fh.write(f"{rec['N']}\t{rec['L']}\t{rec['index']}\t{rec['formula']}\n")

# Save the formulas that hit the timeout, so they can be inspected / rerun.
timed_out = [r for r in all_results if r.get("timed_out")]
with open(TIMEOUTS_FILE, "w") as fh:
    for rec in timed_out:
        fh.write(f"{rec['N']}\t{rec['L']}\t{rec['index']}\t{rec['formula']}\n")

# Save the per-formula stats for all automata (gba + buchi) plus run time as CSV.
fieldnames = [
    "N", "L", "index", "formula", "ok", "error", "is_property",
    "gba_states", "gba_transitions", "buchi_states", "buchi_transitions",
    "run_time_s", "timed_out",
]
with open(STATS_FILE, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(all_results)

# Save everything (stats + per-run times + per-combination times) as JSON.
with open(RESULTS_FILE, "w") as fh:
    json.dump({
        "N_values": N_VALUES,
        "L_values": L_VALUES,
        "F": F,
        "timeout_s": TIMEOUT_S,
        "combination_times_s": {f"N={n},L={L}": t for (n, L), t in combo_times.items()},
        "total_time_s": sum(combo_times.values()),
        "timed_out": [
            {"N": r["N"], "L": r["L"], "index": r["index"], "formula": r["formula"]}
            for r in timed_out
        ],
        "results": all_results,
    }, fh, indent=2)

print(f"Wrote {FORMULAS_FILE}, {STATS_FILE}, {RESULTS_FILE}, {TIMEOUTS_FILE}")
print(f"Total: {len(all_results)} runs, {sum(combo_times.values()):.4f}s, "
      f"{len(timed_out)} timed out")

Wrote formulas.txt, stats.csv, results.json, timeouts.txt
Total: 1200 runs, 36314.9416s, 94 timed out


In [7]:
# Quick per-combination summary.
from statistics import mean

for n in N_VALUES:
    for L in L_VALUES:
        rows = [r for r in all_results if r["N"] == n and r["L"] == L and r["ok"]]
        if not rows:
            print(f"N={n} L={L}: no successful runs")
            continue
        print(
            f"N={n} L={L}: "
            f"gba states avg {mean(r['gba_states'] for r in rows):.2f}, "
            f"buchi states avg {mean(r['buchi_states'] for r in rows):.2f}, "
            f"properties {sum(bool(r['is_property']) for r in rows)}/{len(rows)}, "
            f"run time avg {mean(r['run_time_s'] for r in rows):.4f}s, "
            f"combo time {combo_times[(n, L)]:.4f}s"
        )

N=1 L=5: gba states avg 7.79, buchi states avg 3.49, properties 85/97, run time avg 0.1939s, combo time 10.8821s
N=1 L=10: gba states avg 19.91, buchi states avg 4.73, properties 52/100, run time avg 0.0237s, combo time 0.8234s
N=1 L=15: gba states avg 69.31, buchi states avg 4.86, properties 56/98, run time avg 18.0072s, combo time 1336.8479s
N=1 L=20: gba states avg 134.00, buchi states avg 4.92, properties 48/89, run time avg 35.9272s, combo time 4207.1052s
N=2 L=5: gba states avg 8.80, buchi states avg 4.31, properties 67/100, run time avg 0.0368s, combo time 3.3351s
N=2 L=10: gba states avg 31.37, buchi states avg 5.74, properties 44/100, run time avg 0.0466s, combo time 1.7239s
N=2 L=15: gba states avg 117.46, buchi states avg 7.69, properties 33/96, run time avg 22.8583s, combo time 2119.3072s
N=2 L=20: gba states avg 220.51, buchi states avg 7.96, properties 23/73, run time avg 74.6901s, combo time 9292.0629s
N=3 L=5: gba states avg 8.75, buchi states avg 4.72, properties 56/10